In [37]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [38]:
#load dataset
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [39]:
### Pre Process data
### drop feature
data = data.drop(['RowNumber','CustomerId','Surname'],axis=1)


In [40]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [41]:
#encode categorical value
label_encode_gender = LabelEncoder()
data['Gender'] = label_encode_gender.fit_transform(data['Gender'])
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [42]:
#one Hot encoding for Geography
from sklearn.preprocessing import OneHotEncoder
one_hot_encoder = OneHotEncoder()
geo_encoder = one_hot_encoder.fit_transform(data[['Geography']]).toarray()
geo_encoder

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [45]:
one_hot_encoder.get_feature_names_out(['Geography'])
               

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [46]:
geo_encoder_df = pd.DataFrame(geo_encoder,columns=one_hot_encoder.get_feature_names_out(['Geography']))
geo_encoder_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [ ]:
#concat data abd geo_encode_df

#drop the column
data = data.drop(columns=['Geography'],axis=1)
data


KeyError: "['Geography'] not found in axis"

In [49]:
data = pd.concat([data,geo_encoder_df],axis=1)

In [54]:
# save the encoders
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encode_gender,file)

with open('one_hot_encoder_geography.pkl','wb') as file:
    pickle.dump(one_hot_encoder,file) 


In [55]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [56]:
#divide dataset independent and dependent features
X = data.drop(['Exited'],axis=1)
Y = data['Exited']

In [57]:
# train test split
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=42)

In [59]:
# standardize the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [60]:
X_train
X_test

array([[-0.57749609,  0.91324755, -0.6557859 , ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.29729735,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.52560743, -1.09499335,  0.48508334, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.81311987, -1.09499335,  0.77030065, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.41876609,  0.91324755, -0.94100321, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.24540869,  0.91324755,  0.00972116, ..., -0.99850112,
         1.72572313, -0.57638802]])

In [112]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

## ANN

In [ ]:
#import sequential

import tensorflow as tf
from tensorflow.keras.models import Sequential

In [65]:
#import Dense Early stopping and callback
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,Callback

In [69]:
X_train.shape[1],

(12,)

In [72]:
#initialize the model 
model = Sequential(
    [
        Dense(64,input_shape=(X_train.shape[1],),activation='relu'),
        Dense(32,activation='relu'),
        Dense(1,activation='sigmoid'),
    ]
)

In [73]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [101]:
#compile the model

opt = tf.keras.optimizers.Adam(learning_rate = 0.01)
    
loss = tf.keras.losses.BinaryCrossentropy

model.compile(optimizer=opt,loss="binary_crossentropy",metrics=["accuracy"])

In [102]:
#tensorboard
import datetime
from tensorflow.keras.callbacks import EarlyStopping,Callback,TensorBoard
log_dir = "log/fit"+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensor_flow_board = TensorBoard(log_dir=log_dir,histogram_freq=1)


In [103]:
#early stopping 
early_stop = EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [104]:
#train the model
model.fit(X_train,Y_train,validation_data=(X_test,Y_test),epochs=100,
          callbacks=[tensor_flow_board,early_stop]
          )

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8900 - loss: 0.2413 - val_accuracy: 0.8525 - val_loss: 0.4978
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8934 - loss: 0.2418 - val_accuracy: 0.8530 - val_loss: 0.4864
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8937 - loss: 0.2261 - val_accuracy: 0.8530 - val_loss: 0.5154
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8969 - loss: 0.2269 - val_accuracy: 0.8530 - val_loss: 0.5172
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8888 - loss: 0.2351 - val_accuracy: 0.8515 - val_loss: 0.5033
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8920 - loss: 0.2309 - val_accuracy: 0.8525 - val_loss: 0.5256
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8968 - loss: 0.2329 - val_accuracy: 0.8505 - val_loss: 0.5192
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8977 - loss: 0.2234 - val_accu

In [105]:
#save the model
model.save('model.keras')

In [106]:
%load_ext tensorboard

In [110]:
%tensorboard --logdir log/fit20250316-205654

Reusing TensorBoard on port 6006 (pid 6993), started 0:00:29 ago. (Use '!kill 6993' to kill it.)